# Pokemon processing and within-type ranking

Team Error_105 course project. See the repository README for inputs, working directories and scope.


In [ ]:
import json
import pandas as pd
import numpy as np

import sqlalchemy
from sqlalchemy import create_engine, text

In [ ]:
with open('../../data/pokemon_data/generation_pokemon/generation_pokemon.json', 'r') as file:
    pokemon_data = json.load(file)

poke_df = pd.DataFrame(pokemon_data)
display(poke_df)

In [ ]:
with open('../../data/pokemon_data/species_details/pokemon_1_species_details.json', 'r') as file:
    test_data = json.load(file)

test_df_1 = pd.json_normalize(test_data)
test_df_1.set_index('id', inplace = True)
display(test_df_1)
print(test_df_1.columns)

In [ ]:
with open('../../data/pokemon_data/pokemon_details/pokemon_1_details.json', 'r') as file:
    test_data = json.load(file)

test_df_2 = pd.json_normalize(test_data)
test_df_2.set_index('id', inplace = True)
display(test_df_2)
print(test_df_2.columns)

In [ ]:
print(test_df_2.iloc[0,-7])
print(test_df_2.iloc[0,-6])

In [ ]:
range_of_poke_id = range(poke_df['pokemon_id'].min(), poke_df['pokemon_id'].max()+1)

In [ ]:
def extract_pokemon_species_data(): 
    json_data = {
    pokemon_id: json.load(open(f'../../data/pokemon_data/species_details/pokemon_{pokemon_id}_species_details.json', 'r'))
    for pokemon_id in range_of_poke_id
    }
    
    poke_df['habitat_name'] = poke_df['pokemon_id'].apply(lambda x: json_data[x]['habitat']['name'] if json_data[x]['habitat'] else None)
    poke_df['pokemon_description'] = poke_df['pokemon_id'].apply(lambda x: json_data[x]['flavor_text_entries']['flavor_text'])
    return
    
extract_pokemon_species_data()

In [ ]:
poke_df['pokemon_description'] = poke_df['pokemon_description'].str.replace('\n', ' ', regex=False)

In [ ]:
def extract_pokemon_details_data():
    json_data = {
    pokemon_id: json.load(open(f'../../data/pokemon_data/pokemon_details/pokemon_{pokemon_id}_details.json', 'r'))
    for pokemon_id in range_of_poke_id
    }
    poke_df['type_1'] = poke_df['pokemon_id'].apply(lambda x: json_data[x]['types'][0]['type']['name'])
    poke_df['type_2'] = poke_df['pokemon_id'].apply(lambda x: json_data[x]['types'][1]['type']['name'] if len(json_data[x]['types']) > 1 else None)
    poke_df['pokemon_portrait'] = poke_df['pokemon_id'].apply(lambda x: json_data[x]['pokemon_portrait'])

    stats = ['hp', 'attack', 'defense', 'special_attack', 'special_defense', 'speed']
    for i, stat in enumerate(stats):
        poke_df[f'{stat}_stat'] = poke_df['pokemon_id'].apply(lambda x: json_data[x]['stats'][i]['base_stat'])
        
    return 

extract_pokemon_details_data()

In [ ]:
def calculate_total_stat(df: pd.DataFrame):
    stat_columns = [col for col in df.columns if 'stat' in col]
    df['total_stat'] = df[stat_columns].sum(axis=1)

calculate_total_stat(poke_df)

In [ ]:
poke_df.set_index('pokemon_id', inplace = True)
fire_poke_df = poke_df[poke_df['type_1'] == 'fire']
ice_poke_df = poke_df[poke_df['type_1'] == 'ice']
water_poke_df = poke_df[poke_df['type_1'] == 'water']

In [ ]:
poke_df['total_stat'].value_counts()
poke_df['type_1'].value_counts()

In [ ]:
poke_df.to_json('../../data/pokemon_data/main_pokemon_df.json')

In [ ]:
def assign_locations(df):
    df = df.copy()
    df.sort_values(by=['total_stat', 'hp_stat'], ascending=[False, False], inplace=True)
    df.loc[:,'ranking'] = range(1, len(df) + 1)
    return df

In [ ]:
fire_poke_df = assign_locations(fire_poke_df)
display(fire_poke_df)

In [ ]:
ice_poke_df = assign_locations(ice_poke_df)
display(ice_poke_df)

In [ ]:
water_poke_df = assign_locations(water_poke_df)

In [ ]:
database_path = "../../data/main.db"
engine = create_engine(f"sqlite:///{database_path}")
create_tracks_statement = text("""
CREATE TABLE IF NOT EXISTS fire_pokemon (
    pokemon_id SMALLINT,
    name VARCHAR(20),
    url VARCHAR(50),  
    generation VARCHAR(1),
    habitat_name VARCHAR(20),
    pokemon_portrait VARCHAR(50),                                  
    pokemon_description TEXT,
    type_1 VARCHAR(20),
    type_2 VARCHAR(20),
    hp_stat SMALLINT,
    attack_stat SMALLINT,
    defense_stat SMALLINT,
    special_attack_stat SMALLINT,
    special_defense_stat SMALLINT,
    speed_stat SMALLINT,
    total_stat SMALLINT, 
    ranking SMALLINT PRIMARY KEY
);
""")

with engine.connect() as conn:
    conn.execute(text('DROP TABLE IF EXISTS fire_pokemon;'))
    conn.execute(create_tracks_statement)

fire_poke_df.to_sql("fire_pokemon", engine, if_exists = "append", index = True)

with engine.connect() as conn:
    result = conn.execute(text("SELECT COUNT(*) FROM fire_pokemon"))
    print(f"Number of rows in database: {result.scalar()}")

In [ ]:
database_path = "../../data/main.db"
engine = create_engine(f"sqlite:///{database_path}")
create_tracks_statement = text("""
CREATE TABLE IF NOT EXISTS ice_pokemon (
    pokemon_id SMALLINT,
    name VARCHAR(20),
    url VARCHAR(50),
    generation VARCHAR(1),
    habitat_name VARCHAR(20), 
    pokemon_portrait VARCHAR(50),      
    pokemon_description TEXT,
    type_1 VARCHAR(20),
    type_2 VARCHAR(20),
    hp_stat SMALLINT,
    attack_stat SMALLINT,
    defense_stat SMALLINT,
    special_attack_stat SMALLINT,
    special_defense_stat SMALLINT,
    speed_stat SMALLINT,
    total_stat SMALLINT, 
    ranking SMALLINT PRIMARY KEY
);
""")

with engine.connect() as conn:
    conn.execute(text('DROP TABLE IF EXISTS ice_pokemon;'))
    conn.execute(create_tracks_statement)

ice_poke_df.to_sql("ice_pokemon", engine, if_exists = "append", index = True)

with engine.connect() as conn:
    result = conn.execute(text("SELECT COUNT(*) FROM ice_pokemon"))
    print(f"Number of rows in database: {result.scalar()}")

In [ ]:
database_path = "../../data/main.db"
engine = create_engine(f"sqlite:///{database_path}")
create_tracks_statement = text("""
CREATE TABLE IF NOT EXISTS water_pokemon (
    pokemon_id SMALLINT,
    name VARCHAR(20),
    url VARCHAR(50),  
    generation VARCHAR(1),
    habitat_name VARCHAR(20),
    pokemon_portrait VARCHAR(50),                                  
    pokemon_description TEXT,
    type_1 VARCHAR(20),
    type_2 VARCHAR(20),
    hp_stat SMALLINT,
    attack_stat SMALLINT,
    defense_stat SMALLINT,
    special_attack_stat SMALLINT,
    special_defense_stat SMALLINT,
    speed_stat SMALLINT,
    total_stat SMALLINT, 
    ranking SMALLINT PRIMARY KEY
);
""")

with engine.connect() as conn:
    conn.execute(text('DROP TABLE IF EXISTS water_pokemon;'))
    conn.execute(create_tracks_statement)

water_poke_df.to_sql("water_pokemon", engine, if_exists = "append", index = True)

with engine.connect() as conn:
    result = conn.execute(text("SELECT COUNT(*) FROM water_pokemon"))
    print(f"Number of rows in database: {result.scalar()}")